# **4.3 Triển khai LLM**

## **4.3.1 Setup — Load model, predict 1 case, lấy top features SHAP**

In [ ]:
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt

In [17]:
from dotenv import load_dotenv
load_dotenv()  # đọc file .env, set vào os.environ

True

In [18]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Thêm src vào path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [19]:
# 1. Tìm project_root (chứa folder 'src')
current = Path.cwd().resolve()
project_root = next((p for p in [current, *current.parents] if (p / "src").exists()), None)
if project_root is None:
    raise FileNotFoundError("Không tìm thấy thư mục 'src'")
sys.path.insert(0, str(project_root))
print("project_root =", project_root)

project_root = D:\Python\AIN701_Group_04\demo


In [20]:
# 2. Load model + predict cho 1 hồ sơ vay
from src.models.predict_model import predict_loan, load_model_and_features
from src.models.shap_analysis import get_explainer, get_top_features_for_case

In [21]:
raw_input = {
    'Age': 19, 'Income': 30000, 'LoanAmount': 200000, 'CreditScore': 450,
    'MonthsEmployed': 3, 'NumCreditLines': 5, 'InterestRate': 22.5, 'LoanTerm': 60,
    'DTIRatio': 0.85, 'Education': 'High School', 'EmploymentType': 'Unemployed',
    'MaritalStatus': 'Single', 'HasMortgage': 'No', 'HasDependents': 'No',
    'LoanPurpose': 'Business', 'HasCoSigner': 'No'
}

In [22]:
result, X_input = predict_loan(raw_input)
print(f"Quyết định: {result['status']}  (xác suất = {result['probability_default']:.4f})")

Loaded: XGBClassifier từ xgboost.pkl
Features: 24
Quyết định: Rủi ro cao (Default)  (xác suất = 0.9771)


In [23]:
# 3. Tính SHAP cho đúng case này -> lấy top features (phần đang thiếu trong notebook)
model, feature_names = load_model_and_features()
explainer = get_explainer(model)
shap_values = explainer.shap_values(X_input)
top_features = get_top_features_for_case(shap_values, X_input, idx=0, top_n=5)
print(top_features)

Loaded: XGBClassifier từ xgboost.pkl
Features: 24
          feature     value  shap_value
0             Age      19.0    0.914149
1    InterestRate      22.5    0.641440
2  MonthsEmployed       3.0    0.438928
3          Income   30000.0    0.429357
4      LoanAmount  200000.0    0.339090


## **4.3.2 Thiết kế Prompt**

In [27]:
from src.models.llm_explainer import explain_loan_decision

explanation_text = explain_loan_decision(result, top_features)
print(explanation_text)

Chào bạn, chúng tôi rất tiếc chưa thể phê duyệt yêu cầu này do hệ thống đánh giá khả năng gặp khó khăn khi thanh toán khoản vay của bạn là rất cao (97.7%). Lý do chính là vì bạn còn khá trẻ (19 tuổi) và thời gian làm việc hiện tại còn ngắn (mới được 3 tháng). Thêm vào đó, việc gánh vác một khoản vay lớn trị giá 200.000 với mức lãi suất khá cao là 22.5% sẽ tạo ra áp lực tài chính rất lớn so với mức thu nhập hiện tại là 30.000 của bạn. Rất mong bạn thông cảm và hy vọng sẽ có cơ hội hỗ trợ bạn trong tương lai khi các điều kiện tài chính ổn định hơn.


## **4.3.3 Gọi LLM API**

In [25]:
!pip install -U google-genai

## **4.3.4 Test với nhiều case khác nhau (Approve / Reject / borderline)**

In [28]:
from src.models.llm_explainer import build_explanation_prompt, generate_explanation

test_cases = {
    "Rủi ro cao": {'Age': 19, 'Income': 30000, 'LoanAmount': 200000, 'CreditScore': 450,
                   'MonthsEmployed': 3, 'NumCreditLines': 5, 'InterestRate': 22.5, 'LoanTerm': 60,
                   'DTIRatio': 0.85, 'Education': 'High School', 'EmploymentType': 'Unemployed',
                   'MaritalStatus': 'Single', 'HasMortgage': 'No', 'HasDependents': 'No',
                   'LoanPurpose': 'Business', 'HasCoSigner': 'No'},
    "An toàn": {'Age': 45, 'Income': 150000, 'LoanAmount': 20000, 'CreditScore': 780,
                'MonthsEmployed': 120, 'NumCreditLines': 2, 'InterestRate': 5.5, 'LoanTerm': 36,
                'DTIRatio': 0.15, 'Education': "Master's", 'EmploymentType': 'Full-time',
                'MaritalStatus': 'Married', 'HasMortgage': 'Yes', 'HasDependents': 'Yes',
                'LoanPurpose': 'Auto', 'HasCoSigner': 'Yes'},
}

for label, ri in test_cases.items():
    r, X_i = predict_loan(ri)
    sv = explainer.shap_values(X_i)
    tf = get_top_features_for_case(sv, X_i, idx=0, top_n=5)
    p = build_explanation_prompt(r, tf)
    explanation = generate_explanation(p)
    print(f"Case: {label} ({r['status']})")
    print(explanation)
    print()

Loaded: XGBClassifier từ xgboost.pkl
Features: 24
Case: Rủi ro cao (Rủi ro cao (Default))
Chào bạn, chúng tôi rất tiếc phải thông báo rằng hồ sơ đăng ký của bạn hiện có mức độ rủi ro thanh toán khá cao, lên đến 97.7%. Nguyên nhân chính là do số tiền đề nghị vay (200,000) đang quá lớn so với mức thu nhập hiện tại của bạn (30,000). Bên cạnh đó, việc bạn mới làm việc được 3 tháng, ở độ tuổi 19 và phải chịu mức lãi suất cao (22.5%) cũng tạo ra áp lực tài chính rất lớn để hoàn trả khoản vay. Vì những lý do này, hệ thống chưa thể phê duyệt khoản vay cho bạn tại thời điểm hiện tại.

Loaded: XGBClassifier từ xgboost.pkl
Features: 24
Case: An toàn (An toàn (Non-Default))
Chào bạn, chúng tôi rất vui được thông báo rằng hồ sơ của bạn được đánh giá là vô cùng an toàn với tỷ lệ rủi ro vỡ nợ cực kỳ thấp, chỉ ở mức 0,4%. Quyết định này được đưa ra nhờ vào nền tảng tài chính rất vững chắc của bạn, nổi bật là mức thu nhập cao (150.000) cùng thời gian làm việc lâu dài và ổn định (120 tháng). Thêm vào đó

## **4.3.5 Nhận xét chất lượng giải thích**